# Overview
> In this phase, webscraping amazon is the main goal.

The study shall make use of selenium and BeautifulSoup to create an etl system that can scrape data from amazon.com
and use aws s3 as storage.

## Extract
Here we scrape the data from amazon.com

In [7]:
# Importing dependencies
from selenium import webdriver
import pandas as pd
import numpy as np
from tqdm import tqdm
import random
import time
import warnings
warnings.filterwarnings("ignore")

from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ChromeOptions
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ChromeOptions
from selenium.common.exceptions import TimeoutException, NoSuchElementException

from bs4 import BeautifulSoup
import requests
import re
import boto3
import getpass
import json

import spacy
import logging

In [2]:
url = "https://www.amazon.com/"

In [3]:
# Define a function to set up and initialize the driver
def initialize_driver():
    chrome_options = webdriver.ChromeOptions()
    chrome_driver_path = '/usr/bin/chromedriver'
    driver = webdriver.Chrome(chrome_driver_path, options=chrome_options)
    return driver

# Define a function to perform actions
def perform_actions(driver, url):
    url = url
    driver.get(url)
    
    # Define a wait
    wait = WebDriverWait(driver, 10)
    
    # maximize window
    driver.maximize_window()
    
    try:
        # Your actions here
        all_departments = driver.find_element(By.XPATH, "/html/body/div[1]/header/div/div[4]/div[1]/a")
        all_departments.click()
        
    except NoSuchElementException:
        print("An exception occurred. Restarting the driver and retrying...")
        driver.quit()  # Close the current driver
        driver = initialize_driver()  # Reinitialize the driver
        perform_actions(driver, url)  # Retry the actions

# Initialize the driver
driver = initialize_driver()

# Perform actions
perform_actions(driver, url)

/tmp/ipykernel_14140/251620350.py:5: DeprecationWarning: executable_path has been deprecated, please pass in a Service object
  driver = webdriver.Chrome(chrome_driver_path, options=chrome_options)


In [4]:
# Get all departments
all_deps = driver.find_element(By.XPATH, "//html/body/div[3]/div[2]/div/ul[1]")

# Click see all to see all the departments
see_all = all_deps.find_element(By.XPATH, "//*[@id='hmenu-content']/ul[1]/li[11]/a[1]")
if see_all.text == 'See All':
    see_all.click()

In [5]:
# get the list of all the departments
departments = all_deps.find_elements(By.CLASS_NAME, 'hmenu-item')
department_names = [department.text for department in departments]
department_names

['Digital Content & Devices',
 'Amazon Music',
 'Kindle E-readers & Books',
 'Amazon Appstore',
 'Shop By Department',
 'Electronics',
 'Computers',
 'Smart Home',
 'Arts & Crafts',
 'Automotive',
 'Baby',
 'Beauty and personal care',
 "Women's Fashion",
 "Men's Fashion",
 "Girls' Fashion",
 "Boys' Fashion",
 'Health and Household',
 'Home and Kitchen',
 'Industrial and Scientific',
 'Luggage',
 'Movies & Television',
 'Pet supplies',
 'Software',
 'Sports and Outdoors',
 'Tools & Home Improvement',
 'Toys and Games',
 'Video Games',
 '',
 'See Less',
 'Programs & Features',
 'Gift Cards',
 'Shop By Interest',
 'Amazon Live',
 'International Shopping',
 '',
 'See All',
 '',
 'Help & Settings',
 'Your Account',
 'English',
 'United States',
 'Customer Service',
 'Sign in']

In [6]:
# filter departments
departments_of_interest = department_names[5:27]
departments_to_scrape = departments[5:27]
departments_of_interest

['Electronics',
 'Computers',
 'Smart Home',
 'Arts & Crafts',
 'Automotive',
 'Baby',
 'Beauty and personal care',
 "Women's Fashion",
 "Men's Fashion",
 "Girls' Fashion",
 "Boys' Fashion",
 'Health and Household',
 'Home and Kitchen',
 'Industrial and Scientific',
 'Luggage',
 'Movies & Television',
 'Pet supplies',
 'Software',
 'Sports and Outdoors',
 'Tools & Home Improvement',
 'Toys and Games',
 'Video Games']

In [7]:
# create a dictionary of department names and their respective web elements
deps_dict = dict(zip(departments_of_interest, departments_to_scrape))
deps_dict

{'Electronics': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_116")>,
 'Computers': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_117")>,
 'Smart Home': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_118")>,
 'Arts & Crafts': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_119")>,
 'Automotive': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_120")>,
 'Baby': <selenium.webdriver.remote.webelement.WebElement (session="89daaa87dabc28d6466a7e76a092a752", element="435FEAF32CE638E8727CA7077C6AAC18_element_121")>,
 

In [8]:
print(deps_dict.keys())

dict_keys(['Electronics', 'Computers', 'Smart Home', 'Arts & Crafts', 'Automotive', 'Baby', 'Beauty and personal care', "Women's Fashion", "Men's Fashion", "Girls' Fashion", "Boys' Fashion", 'Health and Household', 'Home and Kitchen', 'Industrial and Scientific', 'Luggage', 'Movies & Television', 'Pet supplies', 'Software', 'Sports and Outdoors', 'Tools & Home Improvement', 'Toys and Games', 'Video Games'])


In [9]:
# # click on department of interest
# deps_dict['Software'].click()

In [10]:
# # department container
# dept_container = driver.find_element(By.XPATH, "/html/body/div[3]/div[2]/div/ul[5]")
# dept_container

<selenium.webdriver.remote.webelement.WebElement (session="dba41103322bb1574ec43daa9cc6e826", element="9E060FDC7BEC957799775CB3C920EC7E_element_152")>

In [20]:
# # department divisions
# dept_divs= dept_container.find_element(By.XPATH, '//*[@id="hmenu-content"]/ul[@class="hmenu hmenu-visible hmenu-translateX"]')

# # Get the division names
# dept_divisions = dept_divs.find_elements(By.CLASS_NAME, 'hmenu-item')
# dept_div_names = [dep.text for dep in dept_divisions]
# print(dept_div_names)

['MAIN MENU', 'Software', 'Accounting & Finance', 'Antivirus & Security', 'Business & Office', "Children's", 'Design & Illustration', 'Digital Software', 'Education & Reference', 'Games', 'Lifestyle & Hobbies', 'Music', 'Networking & Servers', 'Operating Systems', 'Photography', 'Programming & Web Development', 'Tax Preparation', 'Utilities', 'Video']


In [20]:
# def find_dept_divisions(selected_dept, department_dict, driver=driver):
#     """
#     This function comes after the driver has already accessed the department
#     """
#     if selected_dept == 'Electronics':
#         print('valid department')
        
#         department_dict[selected_dept].click()
        
#         # electronics container
#         electronics_container = driver.find_element(By.XPATH, "/html/body/div[3]/div[2]/div/ul[5]")
        
#         # electronics departments
#         elec_deps = electronics_container.find_elements(By.CLASS_NAME, "hmenu-item")
#         elec_dep_names = [dep.text for dep in elec_deps]
        
#         # get the links to the various elements
#         elec_dept_links = []

#         for item in elec_deps[2:]:
#             elec_dept_links.append(item.get_attribute('href'))

#         # create a dictionary
#         elec_dept_dict = dict(zip(elec_dep_names[2:], elec_dept_links))
        
#         return elec_dept_dict
        
#     elif selected_dept in department_dict.keys() and selected_dept != 'Electronics':
#         print('valid department')
        
#         # Click the department
#         department_dict[selected_dept].click()

#         # Get the department container
#         dept_container = driver.find_element(By.XPATH, "/html/body/div[3]/div[2]/div/ul[5]")

#         # department divisions
#         dept_divs= dept_container.find_element(By.XPATH, '//*[@id="hmenu-content"]/ul[@class="hmenu hmenu-visible hmenu-translateX"]')

#         # Get the division names
#         dept_divisions = dept_divs.find_elements(By.CLASS_NAME, 'hmenu-item')
        
#         dept_div_names = [dep.text for dep in dept_divisions]

#         # List to hold division links
#         div_links = []

#         # The first element is 'MAIN MENU', the second is the department name
#         for division in dept_divisions[2:]:
#             div_links.append(division.get_attribute('href'))

#         # Dictionary to capture division name as key and the link as value
#         divs_dict = dict(zip(dept_div_names[2:], div_links))
        
#         return divs_dict

In [9]:
def find_dept_divisions(selected_dept, department_dict, driver=driver):
    """
    This function comes after the driver has already accessed the department
    """
    if selected_dept == 'Electronics':
        print('valid department')
        
        department_dict[selected_dept].click()
        
        # Wait for the electronics container to be present
        electronics_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/div[3]/div[2]/div/ul[5]"))
        )
        
        # Wait for electronics departments to be visible
        elec_deps = WebDriverWait(driver, 10).until(
            EC.visibility_of_all_elements_located((By.CLASS_NAME, "hmenu-item"))
        )
        elec_dep_names = [dep.text for dep in elec_deps]
        
        # Get the links to the various elements
        elec_dept_links = [item.get_attribute('href') for item in elec_deps[2:]]

        # Create a dictionary
        elec_dept_dict = dict(zip(elec_dep_names[2:], elec_dept_links))
        
        return elec_dept_dict
        
    elif selected_dept in department_dict.keys() and selected_dept != 'Electronics':
        print('valid department')
        
        # Click the department
        department_dict[selected_dept].click()

        # Wait for the department container to be present
        dept_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "/html/body/div[3]/div[2]/div/ul[5]"))
        )

        # Wait for department divisions to be visible
        dept_divs = WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.XPATH, '//*[@id="hmenu-content"]/ul[@class="hmenu hmenu-visible hmenu-translateX"]'))
        )

        # Get the division names
        dept_divisions = dept_divs.find_elements(By.CLASS_NAME, 'hmenu-item')
        dept_div_names = [dep.text for dep in dept_divisions]

        # List to hold division links
        div_links = [division.get_attribute('href') for division in dept_divisions[2:]]

        # Dictionary to capture division name as key and the link as value
        divs_dict = dict(zip(dept_div_names[2:], div_links))
        
        return divs_dict

In [10]:
departments_of_interest[6]

'Beauty and personal care'

In [11]:
electronics_dict = find_dept_divisions(departments_of_interest[6], deps_dict)
print(electronics_dict)

valid department
{'Makeup': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A11058281&ref_=nav_em__nav_desktop_sa_intl_makeup_0_2_11_2', 'Skin Care': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A11060451&ref_=nav_em__nav_desktop_sa_intl_skin_care_0_2_11_3', 'Hair Care': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A11057241&ref_=nav_em__nav_desktop_sa_intl__0_2_11_4', 'Fragrance': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A11056591&ref_=nav_em__nav_desktop_sa_intl_fragrance_0_2_11_5', 'Foot, Hand & Nail Care': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A17242866011&ref_=nav_em__nav_desktop_sa_intl_foot_hand_and_nail_care_0_2_11_6', 'Tools & Accessories': 'https://www.amazon.com/s?bbn=16225006011&rh=i%3Aspecialty-aps%2Cn%3A%2116225006011%2Cn%3A11062741&ref_=nav_em__na

In [13]:
# # get the links to the various elements
# elec_dept_links = []

# for item in elec_deps[2:]:
#     elec_dept_links.append(item.get_attribute('href'))

# # create a dictionary
# elec_dept_dict = dict(zip(elec_dep_names[2:], elec_dept_links))
# elec_dept_dict

In [14]:
# driver.get(elec_deps[2].get_attribute('href'))

In [15]:
# acc_product_container = driver.find_element(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div[@class="sg-col-inner"]/span/div[@class="s-main-slot s-result-list s-search-results sg-row"]')

In [16]:
# page_1 = acc_product_container.find_elements(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div/span[1]/div[1]/div/div[@class="sg-col-inner"]')

In [98]:
# item_1 = page_1[0]

In [13]:
# page_html = item_1.get_attribute("outerHTML")
# soup = BeautifulSoup(page_html, 'html.parser')

# soup

In [14]:
# # Access elements using BeautifulSoup
# name = soup.find('h2').find('a').find('span').text
# mean_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-icon-alt').text.split()[0]
# num_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-size-base s-underline-text').text
# price = soup.find('div', class_='a-row a-size-base a-color-base').find('span', class_='a-offscreen').text.replace('$', '').replace('\n', '.')

# print(f"name: {name} \n mean rating: {mean_rating} \n num ratings: {num_rating} \n price: {price}")

In [12]:
def scrape_dept(dept_dict):
    # Dictionary to store items by division
    divisions = {}
    
    # Iterate through each division
    for division_name, division_link in dept_dict.items():
        driver.get(division_link)
        
        # A list to accumulate items from this division
        division_items = []
        
        while True:
            try:
                # Find the product container for the current page
                product_container = driver.find_element(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div[@class="sg-col-inner"]/span/div[@class="s-main-slot s-result-list s-search-results sg-row"]')
                page = product_container.find_elements(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div/span[1]/div[1]/div/div[@class="sg-col-inner"]')
    
                for item in page:
                    try:
                        item_html = item.get_attribute("outerHTML")
                        soup = BeautifulSoup(item_html, 'html.parser')
    
                        item_name = soup.find('h2').find('a').find('span').text
                        item_mean_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-icon-alt').text.split()[0]
                        item_num_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-size-base s-underline-text').text
                        item_price = soup.find('div', class_='a-row a-size-base a-color-base').find('span', class_='a-offscreen').text.replace('$', '').replace('\n', '.')
    
                        division_items.append({
                            "name": item_name,
                            "mean_rating": item_mean_rating,
                            "num_ratings": item_num_rating,
                            "price": item_price
                        })
    
                    except AttributeError:
                        print("Item not found")
    
                try:
                    next_page_element = driver.find_element(By.PARTIAL_LINK_TEXT, "Next")
                    next_page = next_page_element.get_attribute('href')
                    driver.get(next_page)
                    
                except TimeoutException:
                    print("Timeout exception - No 'Next' button found, exiting loop")
                    break
                    
                except NoSuchElementException:
                    print("No 'Next' button found, exiting loop")
                    break
    
            except NoSuchElementException:
                print("No product container found on this page, exiting loop")
                break

        # Store the items from this division in the dictionary with the division name as the key
        divisions[division_name] = division_items

    return divisions

In [16]:
# # Create a dictionary to store items by division
# electronics_divisions = {}

# # Iterate through each division
# for division_name, division_link in elec_dept_dict.items():
#     # Navigate to the division
#     driver.get(division_link)
    
#     # Create a list to accumulate items from this division
#     division_items = []
    
#     while True:
#         try:
#             # Find the product container for the current page
#             acc_product_container = driver.find_element(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div[@class="sg-col-inner"]/span/div[@class="s-main-slot s-result-list s-search-results sg-row"]')
#             page = acc_product_container.find_elements(By.XPATH, '/html/body/div[1]/div[1]/div[1]/div[1]/div/span[1]/div[1]/div/div[@class="sg-col-inner"]')
    
#             for item in page:
#                 try:
#                     item_html = item.get_attribute("outerHTML")
#                     soup = BeautifulSoup(item_html, 'html.parser')
    
#                     item_name = soup.find('h2').find('a').find('span').text
#                     item_mean_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-icon-alt').text.split()[0]
#                     item_num_rating = soup.find('div', class_='a-row a-size-small').find('span', class_='a-size-base s-underline-text').text
#                     item_price = soup.find('div', class_='a-row a-size-base a-color-base').find('span', class_='a-offscreen').text.replace('$', '').replace('\n', '.')
    
#                     division_items.append({
#                         "name": item_name,
#                         "mean_rating": item_mean_rating,
#                         "num_ratings": item_num_rating,
#                         "price": item_price
#                     })
    
#                 except AttributeError:
#                     print("Item not found")
    
#             try:
#                 next_page_element = driver.find_element(By.PARTIAL_LINK_TEXT, "Next")
#                 next_page = next_page_element.get_attribute('href')
#                 driver.get(next_page)
#             except NoSuchElementException:
#                 print("No 'Next' button found, exiting loop")
#                 break
    
#         except NoSuchElementException:
#             print("No product container found on this page, exiting loop")
#             break

#     # Store the items from this division in the dictionary with the division name as the key
#     electronics_divisions[division_name] = division_items

In [13]:
electronics_dept = scrape_dept(electronics_dict)

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
No product container found on this page, exiting loop
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
I

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

...

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
No product container found on this page, exiting loop
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
I

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
No 'Next' button found, exiting loop
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Ite

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not found
Item not f

In [22]:
# Quit the driver
driver.quit()

In [27]:
electronics_dept.keys()

dict_keys(['Accessories & Supplies', 'Camera & Photo', 'Car & Vehicle Electronics', 'Cell Phones & Accessories', 'Computers & Accessories', 'GPS & Navigation', 'Headphones', 'Home Audio', 'Office Electronics', 'Portable Audio & Video', 'Security & Surveillance', 'Service Plans', 'Television & Video', 'Video Game Consoles & Accessories', 'Video Projectors', 'Wearable Technology', 'eBook Readers & Accessories'])

In [33]:
# Check the length of each division
total_items = []

for key, val in electronics_dept.items():
    print(key)
    print(len(electronics_dept[key]))
    key_len = len(electronics_dept[key])
    total_items.append(key_len)
    print("-"*30)
    
print(f"Total:{sum(total_items)} electronics")

Accessories & Supplies
4170
------------------------------
Camera & Photo
2597
------------------------------
Car & Vehicle Electronics
1711
------------------------------
Cell Phones & Accessories
2780
------------------------------
Computers & Accessories
243
------------------------------
GPS & Navigation
3064
------------------------------
Headphones
604
------------------------------
Home Audio
2606
------------------------------
Office Electronics
774
------------------------------
Portable Audio & Video
2140
------------------------------
Security & Surveillance
11070
------------------------------
Service Plans
1
------------------------------
Television & Video
2028
------------------------------
Video Game Consoles & Accessories
188
------------------------------
Video Projectors
1771
------------------------------
Wearable Technology
12263
------------------------------
eBook Readers & Accessories
469
------------------------------
Total:48479 electronics


In [35]:
# upload to s3 to avoid repeating this task
aws_access_key = getpass.getpass("aws access key id here _")
aws_secret_access_key = getpass.getpass("aws secret access key here _")
aws_region = getpass.getpass("aws region here _")

aws access key id here _········
aws secret access key here _········
aws region here _········


In [39]:
# create s3 client
# Create an S3 client
s3_cli = boto3.client('s3',
                  aws_access_key_id=aws_access_key,
                  aws_secret_access_key=aws_secret_access_key)

bucket_name = 'amazon-scraped-products-raw'
s3_cli.create_bucket(Bucket=bucket_name)

ClientError: An error occurred (IllegalLocationConstraintException) when calling the CreateBucket operation: The unspecified location constraint is incompatible for the region specific endpoint this request was sent to.

In [23]:
# store locally to json
file_path = "/data/electronics_dept.json"

# # Write the data to the JSON file
# with open(file_path, 'w') as json_file:
#     json.dump(electronics_dept, json_file)

# print(f'Data has been written to {file_path}')

In [38]:
current_url = driver.current_url
print(current_url)

https://www.amazon.com/s?i=specialty-aps&bbn=16225009011&rh=n%3A%2116225009011%2Cn%3A281407&ref=nav_em__nav_desktop_sa_intl_accessories_and_supplies_0_2_5_2


## Transform
In this section, preprocessing shall take place; cleaning, formatting and aggregating the data.

In [9]:
# clean the data,
# type cast mean rating and num rating to numeric values
file_path = 'data/electronics_dept.json'

with open(file_path, 'r') as f:
    electronics_dept = json.load(f)
    
print(len(electronics_dept))

17


In [10]:
# check the data type
print(type(electronics_dept))
print(electronics_dept.keys())

<class 'dict'>
dict_keys(['Accessories & Supplies', 'Camera & Photo', 'Car & Vehicle Electronics', 'Cell Phones & Accessories', 'Computers & Accessories', 'GPS & Navigation', 'Headphones', 'Home Audio', 'Office Electronics', 'Portable Audio & Video', 'Security & Surveillance', 'Service Plans', 'Television & Video', 'Video Game Consoles & Accessories', 'Video Projectors', 'Wearable Technology', 'eBook Readers & Accessories'])


In [11]:
type(electronics_dept['Accessories & Supplies'])

list

In [12]:
def make_df(dept_dict, dept_name:str):
    df = pd.DataFrame()
    
    dept_name = dept_name
    
    # Iterate trhough each division in the department
    for key, val in dept_dict.items():
        division_df = pd.DataFrame(val)
        division_df['division'] = key
        division_df['department'] = dept_name
        df = df.append(division_df, ignore_index=True)
        
    return df

In [13]:
# create a dataframe for the electronics department
electronics_df = make_df(electronics_dept, 'Electronics')

# preview
electronics_df.head()

,name,mean_rating,num_ratings,price,division,department
0,"IMAXTOP Selfie Light, RGB Video Light with 78 ...",5.0,38,29.99,Accessories & Supplies,Electronics
1,Bose QuietComfort 45 Wireless Bluetooth Noise ...,4.6,"18,437",329.00,Accessories & Supplies,Electronics
2,Ailun Glass Screen Protector for iPhone 15/15 ...,4.5,"5,768",5.98,Accessories & Supplies,Electronics
3,"Bose Headphones 700, Noise Cancelling Bluetoot...",4.5,"34,120",299.00,Accessories & Supplies,Electronics
4,𝟮𝟬𝟮𝟯 𝐔𝐩𝐠𝐫𝐚𝐝𝐞𝐝 for Apple Watch Charger Magnetic...,4.5,"7,292",8.89,Accessories & Supplies,Electronics


### Preliminary Inspection

In [14]:
# metadata
electronics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48479 entries, 0 to 48478
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         48479 non-null  object
 1   mean_rating  48479 non-null  object
 2   num_ratings  48479 non-null  object
 3   price        48479 non-null  object
 4   division     48479 non-null  object
 5   department   48479 non-null  object
dtypes: object(6)
memory usage: 2.2+ MB


In [15]:
print(f"There are {electronics_df.duplicated().sum()} duplicates")

There are 13633 duplicates


### Data Description:
There are 6 columns in this data:
* `name` ~ name of the product
* `mean_rating` ~ mean rating of the product
* `num_ratings` ~ number of users who have rated the product
* `price` ~ price (in USD) of the product
* `division` ~ the amazon department division to which the product belongs e.g wearable technology, accessories & supplies
* `department` ~ the department in which the product belongs e.g electronics, computers, men's fashion etc.

#### Observations:
* All the columns are strings yet some columns like price, mean_rating and num_ratings should be numeric columns
* There are no missing values in this data.
* There are 13633 duplicates in this data.

In [16]:
electronics_df.columns

Index(['name', 'mean_rating', 'num_ratings', 'price', 'division',
       'department'],
      dtype='object')

In [17]:
# Type casting columns to numeric
electronics_df['price'] = electronics_df['price'].str.replace(",", "").astype('float')
electronics_df['mean_rating'] = electronics_df['mean_rating'].astype('float')
electronics_df['num_ratings'] = electronics_df['num_ratings'].str.replace(",", "").astype(int)

# preview
electronics_df.tail()

,name,mean_rating,num_ratings,price,division,department
48474,"Ebook Reader, 7inch TFT LCD E-book Reader, Dig...",2.7,4,66.14,eBook Readers & Accessories,Electronics
48475,TiMOVO Sleeve Case for 6.8 inch All-New Kindle...,5.0,2,13.99,eBook Readers & Accessories,Electronics
48476,kwmobile Cover Compatible with Kobo Libra H2O ...,5.0,3,12.99,eBook Readers & Accessories,Electronics
48477,BoxWave Cable Compatible with Amazon Kindle Oa...,5.0,1,14.95,eBook Readers & Accessories,Electronics
48478,TiMOVO [3 Pack Anti-Glare Screen Protector Des...,2.0,2,10.99,eBook Readers & Accessories,Electronics


In [18]:
# preview
electronics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48479 entries, 0 to 48478
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         48479 non-null  object 
 1   mean_rating  48479 non-null  float64
 2   num_ratings  48479 non-null  int64  
 3   price        48479 non-null  float64
 4   division     48479 non-null  object 
 5   department   48479 non-null  object 
dtypes: float64(2), int64(1), object(3)
memory usage: 2.2+ MB


In [19]:
# Remove duplicates
electronics_df = electronics_df.drop_duplicates()

# preview
electronics_df

,name,mean_rating,num_ratings,price,division,department
0,"IMAXTOP Selfie Light, RGB Video Light with 78 ...",5.0,38,29.99,Accessories & Supplies,Electronics
1,Bose QuietComfort 45 Wireless Bluetooth Noise ...,4.6,18437,329.00,Accessories & Supplies,Electronics
2,Ailun Glass Screen Protector for iPhone 15/15 ...,4.5,5768,5.98,Accessories & Supplies,Electronics
3,"Bose Headphones 700, Noise Cancelling Bluetoot...",4.5,34120,299.00,Accessories & Supplies,Electronics
4,𝟮𝟬𝟮𝟯 𝐔𝐩𝐠𝐫𝐚𝐝𝐞𝐝 for Apple Watch Charger Magnetic...,4.5,7292,8.89,Accessories & Supplies,Electronics
...,...,...,...,...,...,...
48469,BoxWave Case Compatible with Pocketbook Touch ...,5.0,1,25.95,eBook Readers & Accessories,Electronics
48470,PocketBook Cover for InkPad X | Black | PU Lea...,3.9,7,21.13,eBook Readers & Accessories,Electronics
48471,kwmobile Case Compatible with Kobo Libra H2O -...,4.7,5,12.99,eBook Readers & Accessories,Electronics
48472,kwmobile Case Compatible with Kobo Libra 2 Cas...,4.1,13,11.99,eBook Readers & Accessories,Electronics


In [20]:
# preprocess 'name'
electronics_df[electronics_df['name'] == 'Magnetic Wireless Charging Station for Apple Series, 3-in-1 15W Fast Mag-Safe Charger Stand with QC3.0 Adapter, for iPhone 15, 14,13,12 Pro Max/Pro/Mini/Plus, iWatch Ultra/9/8/7/6/5/4/3/2, AirPods\u2026']

,name,mean_rating,num_ratings,price,division,department
365,Magnetic Wireless Charging Station for Apple S...,4.3,9108,39.99,Accessories & Supplies,Electronics
8749,Magnetic Wireless Charging Station for Apple S...,4.3,9108,39.99,Cell Phones & Accessories,Electronics


In [21]:
# Check duplicates one more by subsetting on 'name'
electronics_df['name'].duplicated().sum()

5287

In [22]:
electronics_df = electronics_df.drop_duplicates(subset=['name'])

electronics_df

,name,mean_rating,num_ratings,price,division,department
0,"IMAXTOP Selfie Light, RGB Video Light with 78 ...",5.0,38,29.99,Accessories & Supplies,Electronics
1,Bose QuietComfort 45 Wireless Bluetooth Noise ...,4.6,18437,329.00,Accessories & Supplies,Electronics
2,Ailun Glass Screen Protector for iPhone 15/15 ...,4.5,5768,5.98,Accessories & Supplies,Electronics
3,"Bose Headphones 700, Noise Cancelling Bluetoot...",4.5,34120,299.00,Accessories & Supplies,Electronics
4,𝟮𝟬𝟮𝟯 𝐔𝐩𝐠𝐫𝐚𝐝𝐞𝐝 for Apple Watch Charger Magnetic...,4.5,7292,8.89,Accessories & Supplies,Electronics
...,...,...,...,...,...,...
48469,BoxWave Case Compatible with Pocketbook Touch ...,5.0,1,25.95,eBook Readers & Accessories,Electronics
48470,PocketBook Cover for InkPad X | Black | PU Lea...,3.9,7,21.13,eBook Readers & Accessories,Electronics
48471,kwmobile Case Compatible with Kobo Libra H2O -...,4.7,5,12.99,eBook Readers & Accessories,Electronics
48472,kwmobile Case Compatible with Kobo Libra 2 Cas...,4.1,13,11.99,eBook Readers & Accessories,Electronics


#### Observations:
The study has noted that some characters, while not standard are names of certain products or part of their names e.g 'tn7602pk' is a printer and there are also products like 'samsung s22+' - if the '+' is removed then it would be wrong because there is also a 'samsung s22' and these two items are different. There are also product dimensions like '(49mm)' and '(8.9 in. 11.4 in)' to show the product details. If that is altered then the study shall have removed some of the unique identifiers of these products.

The study will only remove noise (non-standard characters) because if stopwords are removed then products like "in-ear headphones with built-in microphones" will lose a lot of information, lemmatization will also convert a word like "built" to build.

## Text Preprocessing

In [44]:
# declare nlp
nlp = spacy.load("en_core_web_sm")

In [54]:
# Function to remove non-standard characters and lowercase the text
def preprocess_text(text):
    lowercase_text = text.lower()  # Lowercase the text
    cleaned_text = re.sub(r'[^\x00-\x7F]+', '', lowercase_text)  # Remove non-standard characters
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()  # Replace multiple spaces with a single space
    cleaned_text = cleaned_text.strip() # Remove leading and trailing whitespaces
    tokens = re.split(r'(\W+)', cleaned_text)
    text = [token for token in tokens]
    
    return "".join(tokens)

# Inspecting the preprocessing
corpus = electronics_df['name'].apply(preprocess_text)
corpus

0        imaxtop selfie light, rgb video light with 78 ...
1        bose quietcomfort 45 wireless bluetooth noise ...
2        ailun glass screen protector for iphone 15/15 ...
3        bose headphones 700, noise cancelling bluetoot...
4        for apple watch charger magnetic fast charging...
                               ...                        
48469    boxwave case compatible with pocketbook touch ...
48470    pocketbook cover for inkpad x | black | pu lea...
48471    kwmobile case compatible with kobo libra h2o -...
48472    kwmobile case compatible with kobo libra 2 cas...
48476    kwmobile cover compatible with kobo libra h2o ...
Name: name, Length: 29559, dtype: object

### Phrase recognition

In [50]:
# Apply spacy nlp on corpus
docs = corpus.apply(nlp)

# preview
docs

0        (imaxtop, selfie, light, ,, rgb, video, light,...
1        (bose, quietcomfort, 45, wireless, bluetooth, ...
2        (ailun, glass, screen, protector, for, iphone,...
3        (bose, headphones, 700, ,, noise, cancelling, ...
4        (for, apple, watch, charger, magnetic, fast, c...
                               ...                        
48469    (boxwave, case, compatible, with, pocketbook, ...
48470    (pocketbook, cover, for, inkpad, x, |, black, ...
48471    (kwmobile, case, compatible, with, kobo, libra...
48472    (kwmobile, case, compatible, with, kobo, libra...
48476    (kwmobile, cover, compatible, with, kobo, libr...
Name: name, Length: 29559, dtype: object

In [51]:
electronics_df['name']

0        IMAXTOP Selfie Light, RGB Video Light with 78 ...
1        Bose QuietComfort 45 Wireless Bluetooth Noise ...
2        Ailun Glass Screen Protector for iPhone 15/15 ...
3        Bose Headphones 700, Noise Cancelling Bluetoot...
4        𝟮𝟬𝟮𝟯 𝐔𝐩𝐠𝐫𝐚𝐝𝐞𝐝 for Apple Watch Charger Magnetic...
                               ...                        
48469    BoxWave Case Compatible with Pocketbook Touch ...
48470    PocketBook Cover for InkPad X | Black | PU Lea...
48471    kwmobile Case Compatible with Kobo Libra H2O -...
48472    kwmobile Case Compatible with Kobo Libra 2 Cas...
48476    kwmobile Cover Compatible with Kobo Libra H2O ...
Name: name, Length: 29559, dtype: object

## Load
In this section, the study shall load the data to an aws s3 bucket using boto3.